In [1]:
%store -r

In [2]:
import csv
import os
import subprocess

import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

<div class="alert alert-danger">We delete all the data from the DB</div>

In [4]:
session.delete_all()

## Adding the BEL KGs (AD, PD, COVID, CBM) to the Neo4j database

We define a function that imports a BEL KG cypher dump and wires it to a `Collection`/`CollectionEntry`/`BELModel`, then call it once per KG. Every step is scoped to the KG being imported (via a temporary `_NewImport` label and the collection name), so the KGs are not cross-linked.

In [5]:
def save_bel_kg_from_file_path(session, collection_name, input_file_path):
    """Import a BEL KG cypher dump and wire it to a Collection/BELModel.

    Safe to call several times against the same database: every step is
    scoped to the nodes of the KG being imported (via a temporary
    `_NewImport` label and the collection name), so distinct KGs are not
    cross-linked.
    """
    # We import the cypher dump:
    command = [
        "cat",
        str(input_file_path),
        "|",
        "cypher-shell",
        "-a",
        credentials.NEO4J_URI,
        "-u",
        credentials.NEO4J_USERNAME,
        "-p",
        credentials.NEO4J_PASSWORD,
        "-d",
        credentials.NEO4J_DATABASE,
    ]
    result = subprocess.run(" ".join(command), shell=True)
    result.check_returncode()
    # We label the freshly imported nodes as BELModelElement and tag them with
    # a temporary _NewImport label to scope the following steps to this KG only
    # (the NOT n:BELModelElement guard leaves previously imported KGs alone).
    #
    # They are labelled ModelElement as well: a BEL node *is* an element of a
    # model, and the generic helpers in `commute_dm.queries` (get_annotations,
    # get_identifiers, get_subunits, get_ids_and_context) all match
    # `(node:ModelElement)` -- without the label they returned nothing for a BEL
    # node, silently, which is what kept the gene-set analyses of `4_20` to the
    # CellDesigner collections. The label is only matched by those helpers and by
    # one momapy_kb query scoped through `(:LayoutModelMapping)`, which no BEL
    # node is in, so nothing else changes meaning:
    query = """
        MATCH (n)
        WHERE NOT n:Collection AND NOT n:CollectionEntry AND NOT n:BELModel
            AND NOT n:BELModelElement
        SET n:BELModelElement, n:ModelElement, n:_NewImport
        RETURN n
    """
    _ = session.execute_query(query)
    # We make the Collection, CollectionEntry and BELModel nodes:
    query = f"""
        MERGE
            (collection:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(collection_entry:CollectionEntry {{file_path: '{input_file_path}'}})-[:HAS_OBJ]->(model:BELModel)
        RETURN
            collection, collection_entry, model
    """
    _ = session.execute_query(query)
    # We link each newly imported node to this KG's model:
    query = f"""
        MATCH (:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(model:BELModel)
        MATCH (model_element:_NewImport)
        MERGE (model)-[:HAS_NODE]->(model_element)
        RETURN model, model_element
    """
    _ = session.execute_query(query)
    # We extract subgraph information from this KG's relationships and make
    # Subgraph nodes attached to its model:
    query = f"""
        MATCH (:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(model:BELModel)
        MATCH (n:_NewImport)-[r]->(m)
        UNWIND r.annotationSubgraph AS subgraph
        MERGE (model)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph {{name: subgraph}})
        RETURN subgraph_node
    """
    _ = session.execute_query(query)
    # We add this KG's nodes to its subgraphs:
    query = f"""
    MATCH (:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(model:BELModel)
    CALL (model) {{
        MATCH (model)-[:HAS_NODE]->(n)-[r]->(m)
        UNWIND r.annotationSubgraph AS subgraph
        RETURN n AS n, subgraph AS subgraph
        UNION
        MATCH (model)-[:HAS_NODE]->(n)<-[r]-(m)
        UNWIND r.annotationSubgraph AS subgraph
        RETURN n AS n, subgraph AS subgraph
    }}
    MATCH (model)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph)
    WHERE subgraph_node.name = subgraph
    MERGE (subgraph_node)-[:HAS_NODE]->(n)
    RETURN subgraph_node, n
    """
    _ = session.execute_query(query)
    # We make a special "main_model" Subgraph node for this model (so that
    # later queries are uniform for nodes which do not belong to a subgraph):
    query = f"""
        MATCH (:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(model:BELModel)
        MERGE (model)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph {{name: 'main_model'}})
        RETURN subgraph_node
    """
    _ = session.execute_query(query)
    # We add all of this KG's nodes that do not belong to a subgraph to the
    # "main_model" Subgraph node:
    query = f"""
        MATCH (:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(model:BELModel)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph {{name: 'main_model'}})
        MATCH (model)-[:HAS_NODE]->(n:_NewImport)
        WHERE NOT EXISTS {{(n)<-[:HAS_NODE]-(s:Subgraph)}}
        MERGE (subgraph_node)-[:HAS_NODE]->(n)
        RETURN n
    """
    _ = session.execute_query(query)
    # We remove the temporary _NewImport label:
    query = """
        MATCH (n:_NewImport)
        REMOVE n:_NewImport
        RETURN n
    """
    _ = session.execute_query(query)

In [6]:
save_bel_kg_from_file_path(session, "AD_KG_BEL", AD_KG_CYPHER_DATA_FILE)
save_bel_kg_from_file_path(session, "PD_KG_BEL", PD_KG_CYPHER_DATA_FILE)
save_bel_kg_from_file_path(session, "COVID_KG_BEL", COVID_KG_CYPHER_DATA_FILE)
save_bel_kg_from_file_path(session, "CBM_KG_BEL", CBM_KG_CYPHER_DATA_FILE)

## Adding HGNC, UniProt and NCBI Gene annotations to the AD BEL KG

The AD BEL KG encodes its protein abundances by HGNC gene symbol (`p(HGNC:...)`) and carries no cross-references of its own. We attach them to those protein nodes using the same RDF-annotation encoding as the CellDesigner maps in the DB: a per-entry `element_to_annotations` `Mapping` (`Item` &rarr; `Bag` of `RDFAnnotation`), one `RDFAnnotation` per identifier (a single-resource annotation, since alternative identifiers are distinct annotations) qualified by a shared `BQBiol IS` node. That is exactly how a stored species looks — `BCL2` has `hgnc.symbol:BCL2`, `ncbigene:596`, `uniprot:P10415`, `hgnc:990` … as separate `RDFAnnotation`s in one `Bag` — so BEL and CellDesigner nodes read the same way afterwards.

**Four namespaces, not one.** The gene symbol the node is already keyed by gives `hgnc.symbol`, and the HGNC dataset row it resolves to gives `hgnc`, `uniprot_ids` and `entrez_id`. Each analysis needs a different one, and only the first was written before:

- `uniprot` — what `commute_dm.core.get_interface` joins the collections over;
- `ncbigene` — what the **GOAT** analysis of `4_20` builds its gene sets from, since `goat.test_genesets` matches them against the gene lists' `gene` column (the HGNC dataset's `entrez_id`, see `4_00`);
- `hgnc.symbol` — what the **intersection** analysis of `4_20` intersects with the gene lists' `symbol` column.

Coverage is identical across the four: of `AD_KG_BEL`'s 1343 HGNC-namespace proteins, 1332 resolve in the HGNC dataset and all 1332 have both a UniProt and an Entrez id. So this does not change the interface — it only makes the AD side reachable by the gene-set analyses.

This is done only for `AD_KG_BEL`. The function is idempotent: it removes this collection's existing protein annotations before writing, so it can be re-run against a database that already has them without duplicating an `Item` per protein.

In [7]:
def add_hgnc_annotations_to_bel_kg(session, collection_name):
    """Attach HGNC/UniProt/NCBI Gene cross-references to HGNC-encoded BEL proteins.

    Mirrors the RDF-annotation encoding used by the CellDesigner maps in the
    DB: each annotated protein gets an entry in the collection entry's
    `element_to_annotations` Mapping (Item -> Bag of RDFAnnotation), with one
    single-resource RDFAnnotation per identifier (alternative identifiers are
    distinct annotations, not several resources on one) qualified by a shared
    `BQBiol IS` node.

    Four namespaces are written, because the analyses want different ones:
    `uniprot` joins the collections into the interface, `ncbigene` is what GOAT
    matches the gene lists on, `hgnc.symbol` is what the intersection analysis
    uses, and `hgnc` comes along with them. All are looked up in the HGNC dataset
    by the gene symbol the BEL node is already keyed by; only proteins whose
    symbol resolves there are annotated.

    Idempotent: this collection's existing protein annotations are removed first,
    so re-running against an already-annotated database replaces them rather than
    adding a second Item per protein.
    """
    # We build a symbol -> {namespace: [identifier, ...]} map from the HGNC
    # dataset (a TSV). A symbol can map to several (alternative) UniProt ids
    # (pipe-separated); the HGNC id is stored there prefixed ("HGNC:990") while
    # the miriam resource takes it bare, as the CellDesigner maps have it:
    symbol_to_identifiers = {}
    with open(HGNC_DATA_FILE, newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            identifiers = {
                "hgnc.symbol": [row["symbol"]] if row["symbol"] else [],
                "hgnc": ([row["hgnc_id"].split(":")[-1]] if row["hgnc_id"] else []),
                "uniprot": [
                    uniprot_id.strip()
                    for uniprot_id in (row["uniprot_ids"] or "").split("|")
                    if uniprot_id.strip()
                ],
                "ncbigene": [row["entrez_id"]] if row["entrez_id"] else [],
            }
            if any(identifiers.values()):
                symbol_to_identifiers[row["symbol"]] = identifiers
    # We fetch this KG's HGNC-encoded protein nodes:
    query = """
        MATCH (collection:Collection {name: $collection_name})-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(:BELModel)-[:HAS_NODE]->(protein:Protein)
        WHERE protein.namespace = 'HGNC'
        RETURN elementId(protein) AS element_id, protein.name AS symbol
    """
    result = session.execute_query(query, params={"collection_name": collection_name})
    # We resolve each protein's identifiers into the miriam resources to write,
    # skipping symbols without a mapping:
    rows = []
    for record in result:
        identifiers = symbol_to_identifiers.get(record["symbol"])
        if identifiers:
            rows.append(
                {
                    "element_id": record["element_id"],
                    "resources": [
                        f"urn:miriam:{namespace}:{identifier}"
                        for namespace, namespace_identifiers in identifiers.items()
                        for identifier in namespace_identifiers
                    ],
                }
            )
    # We remove this collection's existing protein annotations, so that the
    # function replaces rather than accumulates. Only the Item/Bag/RDFAnnotation
    # nodes go: the Mapping is reused and the shared qualifier is interned by
    # content and may be referenced by the CellDesigner maps:
    query = """
        MATCH (collection:Collection {name: $collection_name})-[:HAS_ENTRY]->(:CollectionEntry)
            -[:HAS_ELEMENT_TO_ANNOTATIONS]->(mapping:Mapping)-[:HAS_ITEM]->(item:Item)
            -[:HAS_VALUE]->(bag:Bag)
        OPTIONAL MATCH (bag)-[:HAS_ITEM]->(annotation:RDFAnnotation)
        DETACH DELETE item, bag, annotation
    """
    _ = session.execute_query(query, params={"collection_name": collection_name})
    # We create the annotation subgraph, matching the CellDesigner-map encoding:
    # one per-entry Mapping and one shared `BQBiol IS` qualifier, then one
    # Item/Bag per protein and one RDFAnnotation per identifier. The Mapping is
    # MERGEd on the entry so re-runs reuse it; the shared qualifier is MERGEd
    # globally (interned by content, as for the CD maps).
    #
    # The `LIMIT 1` is not cosmetic. MERGE acts as a MATCH when the pattern
    # already exists, and binds *every* match -- one row each. Loading the
    # CellDesigner maps leaves a second `BQBiol IS` node in the DB, so on a run
    # against a database that already has them the MERGE yields two rows and
    # every CREATE below happens twice, silently doubling the annotations. It
    # cannot fire on a clean run of this notebook, which annotates before the
    # maps are loaded -- only on the re-run the deletion above exists to allow:
    query = """
        MATCH (collection:Collection {name: $collection_name})-[:HAS_ENTRY]->(entry:CollectionEntry)
        MERGE (entry)-[:HAS_ELEMENT_TO_ANNOTATIONS]->(mapping:Mapping:FrozenDict:BaseNode)
        MERGE (qualifier:BQBiol:BaseNode {name: 'IS', value: 'is'})
        WITH mapping, qualifier ORDER BY elementId(mapping), elementId(qualifier) LIMIT 1
        UNWIND $rows AS row
            MATCH (protein:Protein) WHERE elementId(protein) = row.element_id
            CREATE (mapping)-[:HAS_ITEM]->(item:Item:BaseNode)
            CREATE (item)-[:HAS_KEY]->(protein)
            CREATE (item)-[:HAS_VALUE]->(bag:Bag:FrozenSet:BaseNode)
            WITH bag, qualifier, row
            UNWIND row.resources AS resource
                CREATE (bag)-[:HAS_ITEM]->(annotation:RDFAnnotation:BaseNode {resources: [resource]})
                CREATE (annotation)-[:HAS_QUALIFIER]->(qualifier)
    """
    _ = session.execute_query(
        query, params={"collection_name": collection_name, "rows": rows}
    )

In [8]:
add_hgnc_annotations_to_bel_kg(session, "AD_KG_BEL")

## Adding the COVID-19 DM and PD DM to the Neo4j database

We save the collection to the DB:

In [9]:
collection_names_and_input_file_paths = [
    (
        "COVID_DM_CD",
        COVID_DM_CD_DATA_DIR.glob("*.xml"),
    ),
    (
        "PD_DM_CD",
        PD_DM_CD_DATA_DIR.glob("*.xml"),
    ),
    (
        "COVID_DM_CD_AF",
        COVID_DM_CD_AF_BUILD_DIR.glob("*.xml"),
    ),
    (
        "PD_DM_CD_AF",
        PD_DM_CD_AF_BUILD_DIR.glob("*.xml"),
    ),
]

In [10]:
session.save_collections_from_file_paths(
    collection_names_and_input_file_paths,
    return_type="map",
    with_membership_edges=True,
    integration_mode="hash",
)

/home/rougny/code/momapy/src/momapy/celldesigner/io/celldesigner/reader.py:416: UserWarning: skipping modulation 're49': references a Degraded species (source alias='sa134', target alias='sa131'); Degraded species have no model peer.
  cls._make_and_add_modulation(
/home/rougny/code/momapy/src/momapy/celldesigner/io/celldesigner/reader.py:416: UserWarning: skipping modulation 'ir3e6': references a Degraded species (source alias='csa163', target alias='ir15b'); Degraded species have no model peer.
  cls._make_and_add_modulation(
